# מפגש 19 — פתרון מורה Audited
מקור מעבדת 12 המוצרים, מחלקה חיובית Defect. פתרון זה אינו מיועד למצגת תלמיד.

In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix,accuracy_score,precision_score,recall_score,f1_score,classification_report
actual=["OK"]*8+["Defect"]*4
predicted=["OK"]*5+["Defect"]*3+["OK"]+["Defect"]*3
labels=["OK","Defect"]
cm=confusion_matrix(actual,predicted,labels=labels)
tn,fp,fn,tp=cm.ravel()
p=tp/(tp+fp)
r=tp/(tp+fn)
f=2*tp/(2*tp+fp+fn)
acc=(tp+tn)/cm.sum()

In [ ]:
assert cm.tolist()==[[5,3],[1,3]]
assert (tn,fp,fn,tp)==(5,3,1,3)
assert cm.sum()==12
assert len(actual)==len(predicted)==12
assert tn+fp==8
assert fn+tp==4
assert tp+fp==6
assert tp+fn==4
assert cm.trace()==8
assert abs(p-0.5)<1e-12
assert abs(r-0.75)<1e-12
assert abs(f-0.6)<1e-12
assert abs(acc-2/3)<1e-12

## אמת מול קוד: מוודאים במפורש את המחלקה החיובית.

In [ ]:
assert abs(p-precision_score(actual,predicted,pos_label="Defect"))<1e-12
assert abs(r-recall_score(actual,predicted,pos_label="Defect"))<1e-12
assert abs(f-f1_score(actual,predicted,pos_label="Defect"))<1e-12
assert abs(acc-accuracy_score(actual,predicted))<1e-12
assert abs(f-2*p*r/(p+r))<1e-12
assert abs(f-(p+r)/2)>0.01
print(cm,p,r,f,acc)

## קצה: אין חיזויים חיוביים. 0/0 אינו מוגדר מתמטית; מדיניות zero_division=0 מדווחת אפס.

In [ ]:
edge_actual=["OK"]*8+["Defect"]*2
edge_pred=["OK"]*10
assert accuracy_score(edge_actual,edge_pred)==0.8
assert recall_score(edge_actual,edge_pred,pos_label="Defect")==0
assert precision_score(edge_actual,edge_pred,pos_label="Defect",zero_division=0)==0
assert f1_score(edge_actual,edge_pred,pos_label="Defect",zero_division=0)==0
assert confusion_matrix(edge_actual,edge_pred,labels=labels).tolist()==[[8,0],[2,0]]

## Penguins
הריצו כאן את קוד ההכנה הנתון במקור Teacher-Solution.ipynb (תא הכנה 15): flipper_length_mm, body_mass_g, seed42, Test 25%, K=3, אימון-only scaling. לאחר מכן הפעילו את התאים הבאים. ללא CSV וקוד ההכנה, המחברת אינה Run All עצמאית.

In [ ]:
labels_species=["Adelie","Chinstrap","Gentoo"]
cm_species=confusion_matrix(y_test,predictions,labels=labels_species)
species_table=pd.DataFrame(cm_species,index=labels_species,columns=labels_species)
target="Adelie"
tp_target=species_table.loc[target,target]
fn_target=species_table.loc[target].sum()-tp_target
fp_target=species_table[target].sum()-tp_target
tn_target=cm_species.sum()-tp_target-fn_target-fp_target
assert cm_species.shape==(3,3)
assert cm_species.sum()==len(y_test)
assert cm_species.trace()==sum(y_test.to_numpy()==predictions)
assert tp_target+fn_target==species_table.loc[target].sum()
assert tp_target+fp_target==species_table[target].sum()
assert tp_target+fp_target+fn_target+tn_target==len(y_test)

In [ ]:
p_target=tp_target/(tp_target+fp_target) if tp_target+fp_target else 0
r_target=tp_target/(tp_target+fn_target) if tp_target+fn_target else 0
f_target=2*tp_target/(2*tp_target+fp_target+fn_target) if 2*tp_target+fp_target+fn_target else 0
report=classification_report(y_test,predictions,labels=labels_species,output_dict=True,zero_division=0)
assert abs(report[target]["precision"]-p_target)<1e-12
assert abs(report[target]["recall"]-r_target)<1e-12
assert abs(report[target]["f1-score"]-f_target)<1e-12
assert int(report[target]["support"])==int(species_table.loc[target].sum())
assert sum(int(report[x]["support"]) for x in labels_species)==len(y_test)
print("Adelie",p_target,r_target,f_target)

## הערכה
macro: ממוצע על מחלקות במשקל שווה. weighted: לפי support. F1 אינו מגלם עלויות באופן מפורש. אין לשנות K בגלל תוצאת Test.